# Notebook 08 — Mechanistic Interpretability Depth

**Block 3 from next_steps.md**
- A: Cross-Family Holdout (zero-shot generalisation)
- B: Attention Head Attribution
- C: Systematic SAE Feature Geometry
- D: Representational Similarity Analysis

In [1]:
import warnings, json, random, collections
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.utils.data import TensorDataset, DataLoader
from Bio import SeqIO
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BEST_LAYER = 33
Path('results/mech_interp').mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')

# Load embeddings
tox_embs  = np.load(f'embeddings/natural_toxins_layer{BEST_LAYER}.npy')
ctrl_embs = np.load(f'embeddings/controls_layer{BEST_LAYER}.npy')
rdsg_embs = np.load(f'embeddings/redesigns_layer{BEST_LAYER}.npy')
probe_dir = np.load('results/probe_direction.npy')
main = json.load(open('results/main_results.json'))

# Load sequences
tox_recs  = list(SeqIO.parse('data/toxins_clustered.fasta',  'fasta'))
ctrl_recs = list(SeqIO.parse('data/controls_clustered.fasta','fasta'))
print(f'Tox: {tox_embs.shape} Ctrl: {ctrl_embs.shape} Rdsg: {rdsg_embs.shape}')
print(f'Sequences: {len(tox_recs)} tox, {len(ctrl_recs)} ctrl')

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Device: cuda
Tox: (1712, 1280) Ctrl: (2072, 1280) Rdsg: (643, 1280)
Sequences: 1712 tox, 2072 ctrl


## A — Cross-Family Holdout (Zero-Shot Generalisation)

Hold out one entire toxin family. Retrain probe on remaining. Test on held-out.
Directly answers: *will it work on toxins you haven't seen?*

In [2]:
# Parse toxin families from UniProt description lines
def get_family(rec):
    desc = rec.description.lower()
    for fam in ['conotoxin','phospholipase','snake','scorpion','spider',
                'sea anemone','cone snail','three-finger','kunitz',
                'defensin','hemolysin','neurotoxin']:
        if fam in desc:
            return fam
    return 'other'

families = [get_family(r) for r in tox_recs]
fam_counts = collections.Counter(families)
print('Toxin families:')
for f, c in fam_counts.most_common():
    print(f'  {f:20s}: {c}')

# Pick families with enough members for holdout
holdout_families = [f for f, c in fam_counts.items() if c >= 20 and f != 'other']
print(f'\nFamilies for holdout test: {holdout_families}')

Toxin families:
  other               : 1327
  conotoxin           : 260
  phospholipase       : 39
  neurotoxin          : 32
  snake               : 22
  hemolysin           : 19
  kunitz              : 7
  defensin            : 4
  spider              : 1
  sea anemone         : 1

Families for holdout test: ['neurotoxin', 'phospholipase', 'conotoxin', 'snake']


In [3]:
class ToxinProbe(nn.Module):
    def __init__(self, d=1280):
        super().__init__()
        self.linear = nn.Linear(d, 1)
    def forward(self, x):
        return self.linear(x).squeeze(-1)

def train_probe_quick(X_tr, y_tr, epochs=150):
    probe = ToxinProbe().to(DEVICE)
    ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                       torch.tensor(y_tr, dtype=torch.float32))
    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(probe.parameters(), lr=1e-2, weight_decay=1e-4)
    probe.train()
    for _ in range(epochs):
        for xb, yb in DataLoader(ds, batch_size=256, shuffle=True):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(probe(xb), yb).backward(); opt.step()
    probe.eval()
    return probe

def eval_probe(probe, X_te, y_te):
    with torch.no_grad():
        sc = torch.sigmoid(probe(torch.tensor(X_te, dtype=torch.float32).to(DEVICE))).cpu().numpy()
    return roc_auc_score(y_te, sc), sc

# Run cross-family holdout
print(f'{"Family":<20} {"n_held":>6} {"Train AUROC":>12} {"Holdout AUROC":>14}')
print('-'*56)
holdout_results = {}

for held_fam in holdout_families:
    held_idx = [i for i, f in enumerate(families) if f == held_fam]
    train_idx = [i for i, f in enumerate(families) if f != held_fam]
    
    if len(held_idx) < 10:
        continue
    
    # Training: remaining toxins + all controls
    X_train = np.concatenate([tox_embs[train_idx], ctrl_embs])
    y_train = np.concatenate([np.ones(len(train_idx)), np.zeros(len(ctrl_embs))])
    
    # Test: held-out toxins + subset of controls
    n_ctrl_test = min(len(held_idx)*2, len(ctrl_embs)//2)
    ctrl_test_idx = random.sample(range(len(ctrl_embs)), n_ctrl_test)
    X_test = np.concatenate([tox_embs[held_idx], ctrl_embs[ctrl_test_idx]])
    y_test = np.concatenate([np.ones(len(held_idx)), np.zeros(n_ctrl_test)])
    
    # Also get train-set AUROC for comparison
    Xtr2, Xte2, ytr2, yte2 = train_test_split(X_train, y_train, test_size=0.2,
                                                stratify=y_train, random_state=42)
    
    probe_ho = train_probe_quick(X_train, y_train)
    train_auroc, _ = eval_probe(probe_ho, Xte2, yte2)
    holdout_auroc, holdout_scores = eval_probe(probe_ho, X_test, y_test)
    
    holdout_results[held_fam] = {
        'n_held': len(held_idx),
        'train_auroc': float(train_auroc),
        'holdout_auroc': float(holdout_auroc),
        'mean_score_held': float(holdout_scores[:len(held_idx)].mean()),
    }
    print(f'{held_fam:<20} {len(held_idx):>6} {train_auroc:>12.4f} {holdout_auroc:>14.4f}')

print(f'\nMean holdout AUROC: {np.mean([v["holdout_auroc"] for v in holdout_results.values()]):.4f}')
print('=> Probe generalises zero-shot to unseen toxin families')

Family               n_held  Train AUROC  Holdout AUROC
--------------------------------------------------------
neurotoxin               32       0.9991         0.9951
phospholipase            39       0.9993         0.9987
conotoxin               260       0.9990         0.9994
snake                    22       0.9997         0.9783

Mean holdout AUROC: 0.9929
=> Probe generalises zero-shot to unseen toxin families


## B — Attention Head Attribution

Which attention heads within the key layers (17-20, 29-32) drive toxin discrimination?
Patch out each head, measure DPA drop.

In [4]:
from transformers import EsmForMaskedLM, AutoTokenizer

print('Loading ESM-2 for head attribution...')
tokenizer = AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
esm2 = EsmForMaskedLM.from_pretrained('facebook/esm2_t33_650M_UR50D').to(DEVICE).eval()

N_HEADS = 20  # ESM-2-650M has 20 heads per layer
KEY_LAYERS = [17, 18, 19, 20, 29, 30, 31, 32]
print(f'Key layers: {KEY_LAYERS}, {N_HEADS} heads each')
print(f'Total head ablations: {len(KEY_LAYERS) * N_HEADS}')

Loading ESM-2 for head attribution...


Loading weights: 100%|██████████| 539/539 [00:00<00:00, 6530.88it/s]


Key layers: [17, 18, 19, 20, 29, 30, 31, 32], 20 heads each
Total head ablations: 160


In [5]:
def get_probe_score(seq_str):
    t = tokenizer(seq_str, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = esm2(**t, output_hidden_states=True)
    pooled = out.hidden_states[BEST_LAYER].mean(dim=1)
    w = torch.tensor(probe_dir, dtype=torch.float32).to(DEVICE)
    return float((pooled @ w).item())

def get_layer_dpa(seq_str):
    """Get per-layer DPA for a sequence."""
    t = tokenizer(seq_str, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = esm2(**t, output_hidden_states=True)
    hs = [h.mean(dim=1).squeeze(0).cpu().numpy() for h in out.hidden_states]
    w = probe_dir / (np.linalg.norm(probe_dir) + 1e-8)
    dpa = [float((w * (hs[l] - hs[l-1])).sum()) for l in range(1, 34)]
    return dpa

def ablate_head_score(seq_str, target_layer, target_head):
    """Run ESM-2 with one attention head zeroed out, return probe DPA."""
    t = tokenizer(seq_str, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    
    # Hook to zero out one head's output
    def zero_head_hook(module, input, output):
        # output is (attn_output, attn_weights) or just attn_output
        # ESM-2 attention output shape: (batch, seq_len, embed_dim)
        # Each head contributes embed_dim // n_heads = 64 dims
        head_dim = 1280 // N_HEADS  # 64
        start = target_head * head_dim
        end = start + head_dim
        if isinstance(output, tuple):
            out = list(output)
            out[0] = out[0].clone()
            out[0][:, :, start:end] = 0.0
            return tuple(out)
        else:
            output = output.clone()
            output[:, :, start:end] = 0.0
            return output
    
    # Register hook on the self-attention output projection
    layer_module = esm2.esm.encoder.layer[target_layer]
    hook = layer_module.attention.self.register_forward_hook(zero_head_hook)
    
    with torch.no_grad():
        out = esm2(**t, output_hidden_states=True)
    hook.remove()
    
    hs = [h.mean(dim=1).squeeze(0).cpu().numpy() for h in out.hidden_states]
    w = probe_dir / (np.linalg.norm(probe_dir) + 1e-8)
    dpa_layer = float((w * (hs[target_layer+1] - hs[target_layer])).sum())
    total_score = float((w * hs[BEST_LAYER]).sum())
    return dpa_layer, total_score

In [6]:
# Run head attribution on a sample of toxin sequences
random.seed(42)
sample_tox = random.sample([str(r.seq)[:512] for r in tox_recs], min(10, len(tox_recs)))

print(f'Running head attribution on {len(sample_tox)} toxin sequences...')
print(f'({len(KEY_LAYERS)} layers × {N_HEADS} heads = {len(KEY_LAYERS)*N_HEADS} ablations per seq)\n')

# Baseline DPAs
baseline_dpas = {}
baseline_scores = []
for seq in sample_tox:
    dpa = get_layer_dpa(seq)
    sc = get_probe_score(seq)
    baseline_scores.append(sc)
    for l in KEY_LAYERS:
        baseline_dpas.setdefault(l, []).append(dpa[l-1])

mean_baseline_dpa = {l: np.mean(v) for l, v in baseline_dpas.items()}
print('Baseline DPA per key layer:')
for l in KEY_LAYERS:
    print(f'  Layer {l}: {mean_baseline_dpa[l]:+.2f}')

# Head ablation
head_importance = {}  # (layer, head) -> mean DPA drop

for layer in KEY_LAYERS:
    for head in range(N_HEADS):
        dpa_drops = []
        score_drops = []
        for i, seq in enumerate(sample_tox):
            abl_dpa, abl_score = ablate_head_score(seq, layer, head)
            dpa_drop = baseline_dpas[layer][i] - abl_dpa
            score_drop = baseline_scores[i] - abl_score
            dpa_drops.append(dpa_drop)
            score_drops.append(score_drop)
        
        mean_drop = float(np.mean(dpa_drops))
        mean_score_drop = float(np.mean(score_drops))
        head_importance[(layer, head)] = {
            'dpa_drop': mean_drop,
            'score_drop': mean_score_drop,
            'frac_of_layer': mean_drop / (abs(mean_baseline_dpa[layer]) + 1e-6),
        }
    print(f'  Layer {layer}: done ({N_HEADS} heads)')

# Top heads
ranked = sorted(head_importance.items(), key=lambda x: abs(x[1]['dpa_drop']), reverse=True)
print(f'\nTop 10 attention heads by DPA importance:')
print(f'{"Layer.Head":<12} {"DPA Drop":>10} {"% of Layer":>12} {"Score Drop":>12}')
print('-'*50)
for (l, h), v in ranked[:10]:
    print(f'  {l}.{h:<8} {v["dpa_drop"]:>+10.2f} {v["frac_of_layer"]:>11.1%} {v["score_drop"]:>+12.2f}')

Running head attribution on 10 toxin sequences...
(8 layers × 20 heads = 160 ablations per seq)

Baseline DPA per key layer:
  Layer 17: +2.00
  Layer 18: +0.08
  Layer 19: +2.15
  Layer 20: +3.08
  Layer 29: +0.93
  Layer 30: +4.54
  Layer 31: -0.47
  Layer 32: +3.30
  Layer 17: done (20 heads)
  Layer 18: done (20 heads)
  Layer 19: done (20 heads)
  Layer 20: done (20 heads)
  Layer 29: done (20 heads)
  Layer 30: done (20 heads)
  Layer 31: done (20 heads)
  Layer 32: done (20 heads)

Top 10 attention heads by DPA importance:
Layer.Head     DPA Drop   % of Layer   Score Drop
--------------------------------------------------
  32.11           +25.41      770.8%        +5.20
  32.12           +25.40      770.5%        +5.19
  32.1            +25.39      770.4%        +5.19
  32.4            +25.39      770.3%        +5.19
  32.2            +25.39      770.3%        +5.19
  32.3            +25.39      770.3%        +5.19
  32.13           +25.39      770.3%        +5.19
  32.15      

## C — Systematic SAE Feature Geometry

For each robust feature: what amino acid patterns activate it?

In [7]:
transfer_data = main['sae']['feature_transfer']
robust_feats = [d['feature'] for d in transfer_data if d['transfer_ratio'] > 0.8]
evadable_feats = [d['feature'] for d in transfer_data if d['transfer_ratio'] < 0.3]

# Load SAE activations
try:
    tox_acts = np.load('embeddings/sae_activations_toxins.npy')
    print(f'SAE activations: {tox_acts.shape}')
except FileNotFoundError:
    print('SAE activations not found. Loading SAE...')
    from interplm.sae.inference import load_sae_from_hf
    sae = load_sae_from_hf(plm_model='esm2-650m', plm_layer=BEST_LAYER).to(DEVICE).eval()
    def get_acts(embs, bs=256):
        out = []
        with torch.no_grad():
            for i in range(0, len(embs), bs):
                b = torch.tensor(embs[i:i+bs], dtype=torch.float32).to(DEVICE)
                try: out.append(sae.encode(b).cpu().numpy())
                except: out.append(sae(b)[1].cpu().numpy())
        return np.concatenate(out)
    tox_acts = get_acts(tox_embs)
    np.save('embeddings/sae_activations_toxins.npy', tox_acts)

seqs_arr = [str(r.seq) for r in tox_recs]

# Background AA frequency
ref_counts = collections.Counter()
for s in seqs_arr: ref_counts.update(s)
ref_total = sum(ref_counts.values())
ref_freq = {aa: ref_counts.get(aa,0)/ref_total for aa in 'ACDEFGHIKLMNPQRSTVWY'}

def feature_aa_profile(feat_idx, top_n=30):
    acts = tox_acts[:, feat_idx]
    top_idx = np.argsort(acts)[::-1][:top_n]
    aa_counts = collections.Counter()
    for i in top_idx:
        if i < len(seqs_arr): aa_counts.update(seqs_arr[i])
    total = sum(aa_counts.values()) or 1
    aa_freq = {aa: aa_counts.get(aa,0)/total for aa in 'ACDEFGHIKLMNPQRSTVWY'}
    enrichment = {aa: aa_freq[aa]/(ref_freq.get(aa,1e-4)) for aa in 'ACDEFGHIKLMNPQRSTVWY'}
    top_enriched = sorted(enrichment, key=enrichment.get, reverse=True)[:5]
    return {'enrichment': enrichment, 'top_aa': top_enriched,
            'activation_rate': float((acts > 0).mean())}

print(f'{"Feature":>8} {"Transfer":>9} {"Class":<10} {"Top Enriched AAs":<40} {"Interpretation"}')
print('-'*90)

feature_taxonomy = {}
for d in transfer_data:
    f = d['feature']
    if d['transfer_ratio'] > 0.3 and d['transfer_ratio'] < 0.8:
        continue
    prof = feature_aa_profile(f)
    feature_taxonomy[f] = prof
    top_str = ', '.join(f"{aa}({prof['enrichment'][aa]:.1f}x)" for aa in prof['top_aa'][:3])
    cls = 'ROBUST' if d['transfer_ratio'] > 0.8 else 'EVADABLE'
    aas = prof['top_aa']
    interp = 'Disulfide/Cys-rich' if 'C' in aas[:2] else \
             'Structural rigidity' if any(a in aas[:2] for a in 'GP') else \
             'Charged surface' if any(a in aas[:2] for a in 'KR') else \
             f'Enriched {aas[0]}'
    print(f'{f:>8} {d["transfer_ratio"]:>9.2f} {cls:<10} {top_str:<40} {interp}')

SAE activations: (1712, 10240)
 Feature  Transfer Class      Top Enriched AAs                         Interpretation
------------------------------------------------------------------------------------------
    6122      2.41 ROBUST     P(1.8x), F(1.7x), H(1.1x)                Structural rigidity
    4097      2.64 ROBUST     P(2.2x), K(1.3x), F(1.2x)                Structural rigidity
    5312      0.13 EVADABLE   C(4.8x), W(2.4x), G(1.9x)                Disulfide/Cys-rich
    1055      3.36 ROBUST     E(1.9x), Q(1.6x), V(1.2x)                Enriched E
    9026      0.07 EVADABLE   C(3.2x), H(1.2x), R(1.2x)                Disulfide/Cys-rich
    4397      0.00 EVADABLE   H(1.3x), W(1.3x), A(1.1x)                Enriched H
    9927      0.96 ROBUST     C(3.8x), G(2.0x), Y(1.8x)                Disulfide/Cys-rich
    6971      2.19 ROBUST     K(1.2x), D(1.2x), P(1.2x)                Charged surface
    2704      0.00 EVADABLE   H(1.3x), I(1.2x), A(1.1x)                Enriched H
    197

## D — Representational Similarity Analysis (RSA)

Are toxin/redesign distances in ESM-2 space correlated with each other?
High correlation = ESM-2 tracks structural similarity.

In [8]:
# RSA between toxins and redesigns in ESM-2 embedding space
n_tox = min(100, len(tox_embs))
n_rdsg = min(100, len(rdsg_embs))
n_ctrl = min(100, len(ctrl_embs))

# Pairwise cosine distances
tox_tox = cdist(tox_embs[:n_tox], tox_embs[:n_tox], 'cosine')
tox_rdsg = cdist(tox_embs[:n_tox], rdsg_embs[:n_rdsg], 'cosine')
tox_ctrl = cdist(tox_embs[:n_tox], ctrl_embs[:n_ctrl], 'cosine')
rdsg_ctrl = cdist(rdsg_embs[:n_rdsg], ctrl_embs[:n_ctrl], 'cosine')

print('=== Representational Similarity Analysis ===')
print(f'\nMean cosine distances in ESM-2 embedding space:')
print(f'  Toxin ↔ Toxin:     {tox_tox[np.triu_indices_from(tox_tox,1)].mean():.4f}')
print(f'  Toxin ↔ Redesign:  {tox_rdsg.mean():.4f}')
print(f'  Toxin ↔ Control:   {tox_ctrl.mean():.4f}')
print(f'  Redesign ↔ Control:{rdsg_ctrl.mean():.4f}')

# RSA: correlation between toxin-toxin and toxin-redesign distance matrices
# If high, redesigns preserve the representational geometry of toxins
tox_tox_flat = tox_tox[np.triu_indices_from(tox_tox, 1)]

# Build matched distance matrix: for each toxin pair (i,j),
# compare to mean distance of their redesigns
if n_rdsg >= n_tox:
    rdsg_rdsg = cdist(rdsg_embs[:n_tox], rdsg_embs[:n_tox], 'cosine')
    rdsg_rdsg_flat = rdsg_rdsg[np.triu_indices_from(rdsg_rdsg, 1)]
    rsa_r, rsa_p = spearmanr(tox_tox_flat, rdsg_rdsg_flat)
    print(f'\nRSA (toxin geometry ↔ redesign geometry):')
    print(f'  Spearman r = {rsa_r:.4f}  (p = {rsa_p:.2e})')
    print(f'  => {"Strong" if rsa_r > 0.5 else "Moderate" if rsa_r > 0.3 else "Weak"} '
          f'preservation of representational geometry')
else:
    print(f'\nNot enough redesigns ({n_rdsg}) to match toxins ({n_tox}) for paired RSA')
    rsa_r = None

# Class separability: are toxins/redesigns closer to each other than to controls?
within_tox = tox_rdsg.mean()
between_tox_ctrl = tox_ctrl.mean()
separability = between_tox_ctrl / (within_tox + 1e-8)
print(f'\nClass separability (tox-ctrl dist / tox-rdsg dist): {separability:.2f}')
print(f'  > 1.0 means redesigns are closer to toxins than controls are')

=== Representational Similarity Analysis ===

Mean cosine distances in ESM-2 embedding space:
  Toxin ↔ Toxin:     0.0680
  Toxin ↔ Redesign:  0.0647
  Toxin ↔ Control:   0.1400
  Redesign ↔ Control:0.1453

RSA (toxin geometry ↔ redesign geometry):
  Spearman r = -0.0065  (p = 6.48e-01)
  => Weak preservation of representational geometry

Class separability (tox-ctrl dist / tox-rdsg dist): 2.16
  > 1.0 means redesigns are closer to toxins than controls are


## Save All Results

In [9]:
def _conv(o):
    if isinstance(o, (np.integer, np.floating)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o

all_results = {
    'cross_family_holdout': holdout_results,
    'head_attribution': {
        'top_10_heads': [
            {'layer': l, 'head': h,
             'dpa_drop': v['dpa_drop'],
             'frac_of_layer': v['frac_of_layer'],
             'score_drop': v['score_drop']}
            for (l,h), v in ranked[:10]
        ],
        'n_sequences': len(sample_tox),
    },
    'feature_taxonomy': {str(k): _conv(v) for k, v in feature_taxonomy.items()},
    'rsa': {
        'tox_tox_dist': float(tox_tox[np.triu_indices_from(tox_tox,1)].mean()),
        'tox_rdsg_dist': float(tox_rdsg.mean()),
        'tox_ctrl_dist': float(tox_ctrl.mean()),
        'rsa_r': float(rsa_r) if rsa_r is not None else None,
        'separability': float(separability),
    },
}

with open('results/mech_interp/mech_interp_results.json', 'w') as f:
    json.dump(all_results, f, default=_conv, indent=2)

try:
    main = json.load(open('results/main_results.json'))
    main['mech_interp_depth'] = all_results
    with open('results/main_results.json', 'w') as f:
        json.dump(main, f, default=_conv, indent=2)
    print('Merged into main_results.json')
except: pass

print('\n=== Summary ===')
print(f'Cross-family holdout: mean AUROC = {np.mean([v["holdout_auroc"] for v in holdout_results.values()]):.4f}')
print(f'Top head: L{ranked[0][0][0]}.H{ranked[0][0][1]} (DPA drop={ranked[0][1]["dpa_drop"]:+.2f})')
print(f'RSA separability: {separability:.2f}')
print('Saved → results/mech_interp/')

Merged into main_results.json

=== Summary ===
Cross-family holdout: mean AUROC = 0.9929
Top head: L32.H11 (DPA drop=+25.41)
RSA separability: 2.16
Saved → results/mech_interp/


In [10]:
!zip -r output.zip figures results


  adding: figures/ (stored 0%)
  adding: figures/fig7_attack_convergence.pdf (deflated 29%)
  adding: figures/fig7_attack_convergence.png (deflated 9%)
  adding: figures/fig6_dpa_trajectory.pdf (deflated 31%)
  adding: figures/fig6_dpa_trajectory.png (deflated 8%)
  adding: results/ (stored 0%)
  adding: results/circuits/ (stored 0%)
  adding: results/circuits/dpa_redesign.npy (deflated 43%)
  adding: results/circuits/dpa_control.npy (deflated 43%)
  adding: results/circuits/dpa_toxin.npy (deflated 43%)
  adding: results/circuits/circuit_results.json (deflated 62%)
  adding: results/circuits/layer_auroc_sweep.npy (deflated 48%)
  adding: results/circuits/recovery_natural.npy (deflated 38%)
  adding: results/circuits/recovery_redesign.npy (deflated 37%)
  adding: results/mech_interp/ (stored 0%)
  adding: results/mech_interp/mech_interp_results.json (deflated 69%)
  adding: results/sae_results.json (deflated 74%)
  adding: results/figures/ (stored 0%)
  adding: results/figures/dms_heatm